In [9]:
from sklearn.datasets import load_digits

digits = load_digits()

X = digits.data
y = digits.target

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

models = {
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),

    "Random Forest": RandomForestClassifier(
        random_state=42
    ),

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000))
    ]),

    "Gaussian NB": GaussianNB(),

    "Multinomial NB": MultinomialNB(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    )
}

In [11]:
from sklearn.model_selection import cross_val_score

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    print(name)
    print("Scores:", scores)
    print("Mean Accuracy:", scores.mean())
    print()

SVM
Scores: [0.97569444 0.97916667 0.96864111 0.99303136 0.97212544]
Mean Accuracy: 0.9777318041037553

Random Forest
Scores: [0.97916667 0.96875    0.96515679 0.98606272 0.96864111]
Mean Accuracy: 0.9735554587688734

Logistic Regression
Scores: [0.97569444 0.96875    0.95121951 0.97560976 0.94076655]
Mean Accuracy: 0.9624080526519551

Gaussian NB
Scores: [0.8125     0.78472222 0.85017422 0.87456446 0.82578397]
Mean Accuracy: 0.8295489740611692

Multinomial NB
Scores: [0.875      0.89583333 0.90243902 0.92682927 0.87456446]
Mean Accuracy: 0.8949332171893147

Decision Tree
Scores: [0.85763889 0.86458333 0.82578397 0.86759582 0.83972125]
Mean Accuracy: 0.8510646535036779



In [12]:
from sklearn.model_selection import GridSearchCV

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC())
])

svm_params = {
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["linear", "rbf"],
    "model__gamma": ["scale", "auto"]
}

svm_grid = GridSearchCV(
    svm_pipeline,
    svm_params,
    cv=5,
    scoring="accuracy"
)

svm_grid.fit(X_train, y_train)

print("Best Parameters:")
print(svm_grid.best_params_)

print("Best CV Accuracy:")
print(svm_grid.best_score_)

Best Parameters:
{'model__C': 10, 'model__gamma': 'auto', 'model__kernel': 'rbf'}
Best CV Accuracy:
0.9805216802168022


In [13]:
rf =RandomForestClassifier(random_state=42)

rf_params = {
    "n_estimators" : [50,100,200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    rf,
    rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best Parameters:")
print(rf_grid.best_params_)

print("Best CV Accuracy:")
print(rf_grid.best_score_)

Best Parameters:
{'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Best CV Accuracy:
0.9749516066589237


In [14]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

logistic_params = {
    "model__C": [0.01, 0.1, 1, 10]
}

logistic_grid = GridSearchCV(
    logistic_pipeline,
    logistic_params,
    cv=5,
    scoring="accuracy"
)

logistic_grid.fit(X_train, y_train)

print(logistic_grid.best_params_)
print(logistic_grid.best_score_)

{'model__C': 1}
0.9624080526519551


In [15]:
dt = DecisionTreeClassifier(
    random_state=42
)

dt_params = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid = GridSearchCV(
    dt,
    dt_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

dt_grid.fit(X_train, y_train)

print(dt_grid.best_params_)
print(dt_grid.best_score_)

{'criterion': 'entropy', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
0.8594439605110337


In [17]:
gnb = GaussianNB()

gnb_params = {
    "var_smoothing": [
        1e-11,
        1e-10,
        1e-9,
        1e-8,
        1e-7
    ]
}

gnb_grid = GridSearchCV(
    gnb,
    gnb_params,
    cv=5,
    scoring="accuracy"
)

gnb_grid.fit(X_train, y_train)

print(gnb_grid.best_params_)
print(gnb_grid.best_score_)

{'var_smoothing': 1e-07}
0.8685080332946187


In [16]:
mnb = MultinomialNB()

mnb_params = {
    "alpha": [0.01, 0.1, 0.5, 1, 2, 5]
}

mnb_grid = GridSearchCV(
    mnb,
    mnb_params,
    cv=5,
    scoring="accuracy"
)

mnb_grid.fit(X_train, y_train)

print(mnb_grid.best_params_)
print(mnb_grid.best_score_)

{'alpha': 5}
0.8970189701897018


In [18]:
searches = {
    "SVM": svm_grid,
    "Random Forest": rf_grid,
    "Logistic Regression": logistic_grid,
    "Gaussian NB": gnb_grid,
    "Multinomial NB": mnb_grid,
    "Decision Tree": dt_grid
}

for name, search in searches.items():
    print(
        name,
        "->",
        search.best_score_
    )

SVM -> 0.9805216802168022
Random Forest -> 0.9749516066589237
Logistic Regression -> 0.9624080526519551
Gaussian NB -> 0.8685080332946187
Multinomial NB -> 0.8970189701897018
Decision Tree -> 0.8594439605110337


In [20]:
best_name = max(
    searches,
    key=lambda name: searches[name].best_score_
)

best_search = searches[best_name]

print("Best Model:", best_name)
print("Best CV Accuracy:", best_search.best_score_)
print("Best Parameters:", best_search.best_params_)

best_model = best_search.best_estimator_

y_pred = best_model.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

Best Model: SVM
Best CV Accuracy: 0.9805216802168022
Best Parameters: {'model__C': 10, 'model__gamma': 'auto', 'model__kernel': 'rbf'}
Test Accuracy: 0.9805555555555555
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        28
           2       0.97      1.00      0.99        33
           3       0.97      0.97      0.97        34
           4       0.98      1.00      0.99        46
           5       0.98      0.98      0.98        47
           6       0.97      1.00      0.99        35
           7       1.00      0.94      0.97        34
           8       0.97      0.97      0.97        30
           9       0.97      0.95      0.96        40

    accuracy                           0.98       360
   macro avg       0.98      0.98      0.98       360
weighted avg       0.98      0.98      0.98       360

